<a href="https://colab.research.google.com/github/Defined0101/data/blob/FRS-78-Semisupervised-model/MergedDataset/Bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import nltk
import re
import string
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import classification_report, accuracy_score
import gc

import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# Parquet dosyasının yolu
data_path = '/content/drive/MyDrive/Defined 0101/Data/Cleaned data/foodcom/deduplicated_recipes_foodcom.parquet'  # Buraya kendi dosya yolunuzu yazın

# Parquet dosyasını yükle
df = pd.read_parquet(data_path)

In [17]:
#write df columns
df.columns

Index(['id', 'name', 'description', 'ingredients', 'ingredients_raw_str',
       'serving_size', 'servings', 'steps', 'tags', 'search_terms', 'RecipeId',
       'Name', 'AuthorId', 'AuthorName', 'CookTime', 'PrepTime', 'TotalTime',
       'DatePublished', 'Description', 'Images', 'RecipeCategory', 'Keywords',
       'RecipeIngredientQuantities', 'RecipeIngredientParts',
       'AggregatedRating', 'ReviewCount', 'Calories', 'FatContent',
       'SaturatedFatContent', 'CholesterolContent', 'SodiumContent',
       'CarbohydrateContent', 'FiberContent', 'SugarContent', 'ProteinContent',
       'RecipeServings', 'RecipeYield', 'RecipeInstructions', 'recipe_hash'],
      dtype='object')

In [18]:
df.RecipeCategory.value_counts()

,count
RecipeCategory,
Dessert,60396
Lunch/Snacks,33259
One Dish Meal,30810
Vegetable,26977
Breakfast,20339
...,...
Stir Fry,1
Guatemalan,1
Summer Dip,1


In [19]:
# İlk 5 satırı görüntüle
print(df.head())

# Veri boyutu
print(f"Veri seti boyutu: {df.shape}")

# Kategorilerin dağılımı
print(df['RecipeCategory'].value_counts())


       id                                   name  \
0   96313            grilled garlic cheese grits   
1  232037  simple shrimp and andouille jambalaya   
2   41090               blackandwhite bean salad   
3   60656             crock pot italian zucchini   
4  178298             crock pot italian zucchini   

                                         description  \
0  We love grits, this is another good way to ser...   
1  Simple, easy and very tasty for when you are i...   
2                                               None   
3  This is a good recipe for weight watchers. It ...   
4  A good recipe for zucchini with simple on-hand...   

                                         ingredients  \
0   water grits salt cheddar cheese garlic olive oil   
1  onion red bell pepper garlic cloves large shri...   
2  white beans canned black beans tomatoes onion ...   
3  zucchini yellow squash diced tomatoes onion ga...   
4  zucchini onions diced tomatoes italian salad d...   

             

In [20]:
# Eksik değerlerin sayısı
print(df.isnull().sum())

# MainCategory sütunundaki eksik değerlerin oranı
missing_categories = df['RecipeCategory'].isnull().sum()
total_samples = len(df)
print(f"Etiketi olmayan tarif sayısı: {missing_categories}")
print(f"Toplam tarif sayısı: {total_samples}")
print(f"Etiketi olmayan tariflerin oranı: {missing_categories / total_samples * 100:.2f}%")


id                                 0
name                               0
description                     9597
ingredients                        0
ingredients_raw_str                0
serving_size                       0
servings                           0
steps                              0
tags                               0
search_terms                       0
RecipeId                           0
Name                               0
AuthorId                           0
AuthorName                         0
CookTime                       76265
PrepTime                           0
TotalTime                          0
DatePublished                      0
Description                        5
Images                             0
RecipeCategory                   560
Keywords                           0
RecipeIngredientQuantities         0
RecipeIngredientParts              0
AggregatedRating              209666
ReviewCount                   204423
Calories                           0
F

In [21]:
# nltk paketindeki 'stopwords' listesini kullanmak için indirme işlemi
nltk.download('stopwords')
from nltk.corpus import stopwords

# Önce stopwords listesini ve diğer sabitleri tanımlayalım
stop_words = set(stopwords.words('english'))
stop_words_regex = r'\b(' + r'|'.join(map(re.escape, stop_words)) + r')\b'
punctuation_regex = '[' + re.escape(string.punctuation) + ']'

# Metin ön işleme fonksiyonları
def clean_text_series(series):
    series = series.fillna('').astype(str)
    series = series.str.lower()
    series = series.str.replace(punctuation_regex, '', regex=True)
    series = series.str.replace(r'\d+', '', regex=True)
    series = series.str.replace(stop_words_regex, '', regex=True)
    series = series.str.replace(r'\s+', ' ', regex=True)
    return series.str.strip()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [24]:
# 'ingredients' listesini metin haline getirme ve temizleme
def ingredients_to_text_series(ingredients_series):
    ingredient_names = ingredients_series.apply(
        lambda x: ' '.join([ingredient.get('name', '') for ingredient in x]) if isinstance(x, list) else ''
    )
    return clean_text_series(ingredient_names)

# 'instructions' sütununu temizleme
df['clean_instructions'] = clean_text_series(df['RecipeInstructions'])

# 'desc' sütununu temizleme
df['clean_desc'] = clean_text_series(df['description'])

# 'ingredients' sütununu temizleme
df['clean_ingredients'] = ingredients_to_text_series(df['ingredients'])


In [25]:
df['Result'] = np.nan

nan_indices = df.sample(frac=0.3, random_state=42).index
df.loc[nan_indices, 'Result'] = df.loc[nan_indices, 'RecipeCategory']
df.loc[nan_indices, 'RecipeCategory'] = np.nan

<ipython-input-25-327c5d5fe8f2>:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['Savory Pies' 'Gelatin' 'Camping' ... 'Lunch/Snacks' 'Lunch/Snacks'
 'Dessert']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[nan_indices, 'Result'] = df.loc[nan_indices, 'RecipeCategory']


In [26]:
# Etiketli veriler
labeled_data = df[df['RecipeCategory'].notnull()].copy()

# Etiketsiz veriler
unlabeled_data = df[df['RecipeCategory'].isnull()].copy()


In [27]:
# Özelliklerin birleştirilmesi
labeled_data['text'] = labeled_data['clean_instructions'] + ' ' + labeled_data['clean_desc'] + ' ' + labeled_data['clean_ingredients']
unlabeled_data['text'] = unlabeled_data['clean_instructions'] + ' ' + unlabeled_data['clean_desc'] + ' ' + unlabeled_data['clean_ingredients']


In [28]:
# GPU kullanılabilirliğini kontrol edin
print("GPU Mevcut mu:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Adı:", torch.cuda.get_device_name(0))
else:
    print("GPU kullanılamıyor. CPU kullanılacak.")

GPU Mevcut mu: True
GPU Adı: Tesla T4


In [31]:
# Eğitim ve doğrulama verilerini ayırma
train_texts, val_texts, train_labels, val_labels = train_test_split(
    labeled_data['text'], labeled_data['RecipeCategory'], test_size=0.2, random_state=42
)

# Tüm benzersiz etiketleri topla
all_labels = set(train_labels).union(set(val_labels))

# LabelEncoder'ı tüm etiketlerle eğit
label_encoder = LabelEncoder()
label_encoder.fit(list(all_labels))

# Verileri encode et
train_labels = label_encoder.transform(train_labels)
val_labels = label_encoder.transform(val_labels)

In [32]:
# Tokenizer yükleme
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Veriyi tokenize etme
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [33]:
# Dataset oluşturma
class RecipeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = RecipeDataset(train_encodings, train_labels)
val_dataset = RecipeDataset(val_encodings, val_labels)

In [34]:
# Modeli GPU'ya yükleyin
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_encoder.classes_)
).to(device)

# Eğitim ayarları
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/GradProject/results',  # Sonuçları Google Drive'a kaydetmek için
    evaluation_strategy="epoch",                 # Her epoch sonunda değerlendirme
    per_device_train_batch_size=16,              # GPU için uygun batch boyutu
    per_device_eval_batch_size=16,
    num_train_epochs=3,                          # Eğitim epoch sayısı
    save_strategy="epoch",                       # Her epoch sonunda modeli kaydet
    logging_dir='/content/drive/MyDrive/GradProject/logs',   # Loglar için Drive dizini
    logging_steps=10,                            # Loglama sıklığı
    report_to="none",                            # WandB veya başka bir platforma rapor göndermeyi devre dışı bırak
)

# Trainer nesnesi
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [35]:
# Model eğitimi
trainer.train()

# Modeli ve tokenizer'ı kaydet
model.save_pretrained('/content/drive/MyDrive/GradProject/bert_model')
tokenizer.save_pretrained('/content/drive/MyDrive/GradProject/bert_model')

Epoch,Training Loss,Validation Loss
1,1.980600,1.862669
2,1.454700,1.724990
3,1.492500,1.690698


('/content/drive/MyDrive/GradProject/bert_model/tokenizer_config.json',
 '/content/drive/MyDrive/GradProject/bert_model/special_tokens_map.json',
 '/content/drive/MyDrive/GradProject/bert_model/vocab.txt',
 '/content/drive/MyDrive/GradProject/bert_model/added_tokens.json')